# Build `mayankdate.github.io`

Run every cell top-to-bottom whenever you want to update the site.

What this notebook does:

1. Reads the four YAML files in `config/`.
2. Fetches your public works from ORCID.
3. Applies manual overrides & dedupes preprints that already have a published version.
4. Pulls your Substack posts if the `substack.url` key is set in `profile.yaml`.
5. Generates three QR codes and a `.vcf` file into `assets/qr/`.
6. Renders `index.html`, `projects.html`, `research.html`, `contact.html` at the repo root.

After this notebook runs, `git add . && git commit && git push` publishes the update.

## 1. Make sure the repo root is importable

In [9]:
from pathlib import Path
import sys

# Resolve the repo root from the notebook's location (scripts/build.ipynb)
ROOT = Path.cwd()
if ROOT.name == "scripts":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))
print(f"Repo root: {ROOT}")

Repo root: c:\Mayank's Data\CodingProjects_Personal\Online_Profile


## 2. Load the library and configs

In [10]:
from lib import build as build_mod

cfg = build_mod.load_configs(ROOT)
print(f"Loaded profile for {cfg['profile']['name']}")
print(f"ORCID iD: {cfg['profile']['orcid_id']}")
print(f"Projects: {len(cfg['projects']['projects'])}")
print(f"Manual publications: {len(cfg['publications'].get('manual') or [])}")

Loaded profile for Mayank Date
ORCID iD: 0009-0001-0863-5428
Projects: 10
Manual publications: 8


## 3. Fetch ORCID (sanity check)

In [11]:
from lib import orcid

orcid_pubs = orcid.fetch_orcid_works(cfg['profile']['orcid_id'])
print(f"Fetched {len(orcid_pubs)} work(s) from ORCID.")
for p in sorted(orcid_pubs, key=lambda p: -(p.year or 0))[:5]:
    print(f"  {p.year} · {p.type:20s} · {p.title[:80]}")

Fetched 7 work(s) from ORCID.
  2026 · journal-article      · Feasibility and Validity of Mobile Phone‐Based Proxy Full Pregnancy History (mPF
  2026 · preprint             · Strengthening the Measurement of Digital Inclusion Through Artificial Intelligen
  2026 · journal-article      · Using Large Language Models to Summarize Evidence in Biomedical Articles: Explor
  2026 · journal-article      · Towards digital balance. A data-driven scoping review of the research landscape 
  2025 · journal-article      · Stakeholder perspectives on digital wellbeing in Saudi Arabia: a cross-sectional


## 4. Build the site

In [12]:
build_mod.render_all(ROOT, cfg)

Fetching publications...
  ORCID: fetched 7 work(s)
Building QR codes & vCard...
Grouping projects...
Fetching Substack posts (if configured)...
Rendering sparkline...
Rendering templates...
Build complete.


## 5. (Optional) Preview locally

Open a terminal in the repo root and run:

```bash
python -m http.server 8000
```

Then visit <http://localhost:8000> to preview before pushing.

## 6. Push to GitHub

```bash
git add .
git commit -m "update site"
git push
```

GitHub Pages picks up the change within a minute or two and serves it at <https://mayankdate.github.io/>.